# Phase 1 — Setup & Ingestion

Creates the database schema, seeds the 5-company watchlist, then for each company: finds its FY23/FY24 10-K accession numbers, fetches + parses + segments + chunks + embeds the filing text, and extracts XBRL facts.

**Before running, once per machine:**
```bash
docker compose up -d        # starts local Postgres + pgvector
pip install -e ".[dev,retrieval]"
cp .env.example .env        # fill in GROQ_API_KEY / GOOGLE_AI_API_KEY / DEEPSEEK_API_KEY — DATABASE_URL already matches docker-compose.yml
```
This notebook only needs internet access to reach `sec.gov` and `data.sec.gov` — no cloud database, no Kaggle secrets.

In [1]:
from stance.config import settings
from stance.db import get_engine, init_schema
from stance.seed import seed_all, WATCHLIST, FISCAL_YEARS
from stance.ingest import acquire, parse, segment, chunk, embed, facts
from sqlalchemy import text
import uuid

print(f"Connecting to {settings.database_url.split('@')[-1]}")  # host/db only, never prints the password
engine = get_engine()

Connecting to localhost:5432/stance


## 1a. Schema + seed data

In [2]:
init_schema(engine)
seed_all(engine)

with engine.connect() as conn:
    n_companies = conn.execute(text("SELECT COUNT(*) FROM companies")).scalar()
    n_aliases = conn.execute(text("SELECT COUNT(*) FROM fact_aliases")).scalar()
print(f"companies={n_companies}, fact_aliases={n_aliases}")
assert n_companies == 5
assert n_aliases >= 9

# Re-run this whole notebook a second time if in doubt -- schema and seed
# are idempotent, so the counts above must stay identical.
print(f"Ingesting {len(WATCHLIST)} companies x {FISCAL_YEARS}")

companies=5, fact_aliases=9
Ingesting 5 companies x [2023, 2024]


## 1b. Acquire — find accession numbers, fetch filing HTML, fetch companyfacts

In [3]:
filings_to_process = []  # (cik, ticker, accession_no, fiscal_year, html)
company_facts_cache = {}

for company in WATCHLIST:
    cik, ticker = company["cik"], company["ticker"]
    print(f"--- {ticker} ---")

    submissions = acquire.fetch_submissions(cik)
    target_filings = acquire.find_10k_accessions(submissions, FISCAL_YEARS)
    print(f"  found {len(target_filings)} 10-K filing(s): {[f['accession_no'] for f in target_filings]}")

    for f in target_filings:
        html = acquire.fetch_filing_document(cik, f["accession_no"], f["primary_doc"])
        filings_to_process.append({**f, "cik": cik, "ticker": ticker, "html": html})

        with engine.begin() as conn:
            conn.execute(text(
                "INSERT INTO filings (accession_no, cik, form_type, fiscal_year, filing_date, primary_doc_url, ingest_status) "
                "VALUES (:accession_no, :cik, :form_type, :fiscal_year, :filing_date, :primary_doc, 'acquired') "
                "ON CONFLICT (accession_no) DO NOTHING"
            ), {**f, "cik": cik})

    company_facts_cache[cik] = acquire.fetch_company_facts(cik)

print(f"\nTotal filings to parse: {len(filings_to_process)}")
print(f"Raw HTML cached under: {acquire.RAW_DIR}")

--- AAPL ---
  found 2 10-K filing(s): ['0000320193-24-000123', '0000320193-23-000106']
--- MSFT ---
  found 2 10-K filing(s): ['0000950170-24-087843', '0000950170-23-035122']
--- JPM ---
  found 2 10-K filing(s): ['0000019617-25-000270', '0000019617-24-000225']
--- XOM ---
  found 2 10-K filing(s): ['0000034088-25-000010', '0000034088-24-000018']
--- PFE ---
  found 2 10-K filing(s): ['0000078003-25-000054', '0000078003-24-000039']

Total filings to parse: 10
Raw HTML cached under: /home/aditya-siraskar/Desktop/stance/data/raw


## 1c. Parse, segment, chunk, embed

In [4]:
total_chunks_inserted = 0

for f in filings_to_process:
    print(f"Parsing {f['ticker']} {f['accession_no']}...")
    text_content = parse.parse_filing_html(f["html"])
    sections = segment.segment_filing(text_content)
    print(f"  segments found: {[s.item_code for s in sections]}")

    all_chunks = []
    for section in sections:
        section_id = f"sec-{uuid.uuid4().hex[:12]}"
        with engine.begin() as conn:
            conn.execute(text(
                "INSERT INTO sections (section_id, accession_no, item_code, title, text) "
                "VALUES (:sid, :acc, :item, :title, :text) ON CONFLICT (section_id) DO NOTHING"
            ), {"sid": section_id, "acc": f["accession_no"], "item": section.item_code,
                "title": section.title, "text": section.text})

        section_chunks = chunk.chunk_section(section.text)
        for c in section_chunks:
            all_chunks.append({
                "chunk_id": c.chunk_id, "section_id": section_id, "text": c.text,
                "accession_no": f["accession_no"], "cik": f["cik"], "ticker": f["ticker"],
                "fiscal_year": f["fiscal_year"], "item_code": section.item_code,
            })

    if not all_chunks:
        continue
    embeddings = embed.embed_texts([c["text"] for c in all_chunks])
    with engine.begin() as conn:
        for c, vec in zip(all_chunks, embeddings):
            conn.execute(text(
                "INSERT INTO chunks (chunk_id, section_id, accession_no, cik, ticker, fiscal_year, "
                "item_code, text, embedding, embed_model, chunker_version) "
                "VALUES (:chunk_id, :section_id, :accession_no, :cik, :ticker, :fiscal_year, "
                ":item_code, :text, :embedding, :embed_model, :chunker_version) "
                "ON CONFLICT (chunk_id) DO NOTHING"
            ), {**c, "embedding": str(vec), "embed_model": "bge-small-en-v1.5", "chunker_version": "chunk-v1"})
    total_chunks_inserted += len(all_chunks)
    print(f"  inserted {len(all_chunks)} chunks")

print(f"\nTotal chunks inserted: {total_chunks_inserted}")

Parsing AAPL 0000320193-24-000123...


/home/aditya-siraskar/Desktop/stance/stance/ingest/parse.py:35: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


  segments found: ['1A', '7', '7A', '8']


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  inserted 69 chunks
Parsing AAPL 0000320193-23-000106...


/home/aditya-siraskar/Desktop/stance/stance/ingest/parse.py:35: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


  segments found: ['1A', '7', '7A', '8']
  inserted 72 chunks
Parsing MSFT 0000950170-24-087843...
  segments found: ['7']
  inserted 176 chunks
Parsing MSFT 0000950170-23-035122...
  segments found: ['7']
  inserted 169 chunks
Parsing JPM 0000019617-25-000270...
  segments found: ['1A', '7', '7A', '8']
  inserted 910 chunks
Parsing JPM 0000019617-24-000225...
  segments found: ['1A', '7', '7A', '8']
  inserted 914 chunks
Parsing XOM 0000034088-25-000010...
  segments found: ['1A', '7', '7A', '8']
  inserted 265 chunks
Parsing XOM 0000034088-24-000018...
  segments found: ['1A', '7', '7A', '8']
  inserted 200 chunks
Parsing PFE 0000078003-25-000054...
  segments found: ['1A', '7', '7A', '8']
  inserted 422 chunks
Parsing PFE 0000078003-24-000039...
  segments found: ['1A', '7', '7A', '8']
  inserted 409 chunks

Total chunks inserted: 3606


## 1d. XBRL facts

In [5]:
from stance.seed import FACT_ALIASES

element_names = list({alias[1] for alias in FACT_ALIASES})
total_facts_inserted = 0

for company in WATCHLIST:
    cik = company["cik"]
    # Guard against EDGAR's companyfacts API including comparative figures
    # tagged under a filing we never actually ingested (e.g. a high
    # filing-volume company's older 10-K aged out of the 'recent' window) --
    # only keep facts whose accession is one we actually have a filings row for.
    known_accessions = {f["accession_no"] for f in filings_to_process if f["cik"] == cik}
    extracted = facts.extract_facts(
        company_facts_cache[cik], cik, element_names, FISCAL_YEARS,
        known_accessions=known_accessions,
    )
    inserted = facts.store_facts(engine, extracted)
    total_facts_inserted += inserted
    print(f"{company['ticker']}: extracted {len(extracted)}, inserted {inserted}")

print(f"\nTotal facts inserted: {total_facts_inserted}")

AAPL: extracted 10, inserted 0
MSFT: extracted 10, inserted 0
JPM: extracted 8, inserted 0
XOM: extracted 10, inserted 10
PFE: extracted 11, inserted 11

Total facts inserted: 21


## Exit criteria check

In [6]:
with engine.connect() as conn:
    n_chunks = conn.execute(text("SELECT COUNT(*) FROM chunks")).scalar()
    n_facts = conn.execute(text("SELECT COUNT(*) FROM xbrl_facts")).scalar()
    n_filings = conn.execute(text("SELECT COUNT(*) FROM filings")).scalar()
print(f"filings={n_filings}, chunks={n_chunks}, xbrl_facts={n_facts}")
assert n_chunks > 0 and n_facts > 0, "Ingestion produced no data — check the acquire/parse cells above."

# Hand-check: Apple's FY24 revenue should be a 12-digit number in the ~380-400B range.
from stance.numeric.resolver import resolve_fact
aapl_rev = resolve_fact(engine, "revenue", "0000320193", 2024)
print(f"AAPL FY24 revenue fact: {aapl_rev['value']} {aapl_rev['unit']} (fact_id={aapl_rev['fact_id']})")

filings=10, chunks=7170, xbrl_facts=49
AAPL FY24 revenue fact: 391035000000 usd (fact_id=fact-122fe1a634c63bb7)
